# Use `Lale`, `AIF360` and `DisparateImpactRemover` to mitigate bias for credit risk AutoAI model

This notebook contains the steps and code to demonstrate support of AutoAI experiments in watsonx.ai Runtime service. It introduces commands for bias detecting and mitigation performed with `lale.lib.aif360` module. 

Some familiarity with Python is helpful. This notebook uses Python 3.12.

**NOTE:** The notebook is a continuation for sample notebook: <a href="https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/experiments/autoai/Use%20AutoAI%20to%20train%20fair%20models.ipynb" target="_blank" rel="noopener no referrer">"Use AutoAI to train fair models"</a>. 

## Contents

This notebook contains the following parts:

1. [Set up the environment](#1.-Set-up-the-environment)
2. [Load historical experiment](#2.-Load-historical-experiment)
3. [Bias detection and mitigation](#3.-Bias-detection-and-mitigation)
4. [Refinery with lale](#4.-Refinery-with-lale)
5. [Deploy and score](#5.-Deploy-and-score)
6. [Cleanup](#6.-Cleanup)
7. [Summary and next steps](#7.-Summary-and-next-steps)

<a id="1.-Set-up-the-environment"></a>
## 1. Set up the environment

If you are not familiar with <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> and AutoAI experiments please read more about it in the sample notebook: <a href="https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/experiments/autoai/fairness/Use%20AutoAI%20to%20train%20fair%20models.ipynb" target="_blank" rel="noopener no referrer">"Use AutoAI to train fair models"</a>. 

### Install and import the `ibm-watsonx-ai`, `lale` ,`aif360` and dependencies.
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install wget | tail -n 1
%pip install -U ibm-watsonx-ai | tail -n 1
%pip install "setuptools<81" | tail -n 1 # Needed for `pkg_resources` package
%pip install "snapml>=1.16.4,<1.17.0" | tail -n 1
%pip install "autoai-libs>=3.0.10,<4.0.0" | tail -n 1
%pip install "lale[fairness]>=0.9.2,<0.10.0" | tail -n 1

### Connection to watsonx.ai Runtime

Authenticate the watsonx.ai Runtime service on IBM Cloud. You need to provide platform `api_key` and instance `location`.

You can use [IBM Cloud CLI](https://cloud.ibm.com/docs/cli/index.html) to retrieve platform API Key and instance location.

API Key can be generated in the following way:
```
ibmcloud login
ibmcloud iam api-key-create API_KEY_NAME
```

In result, get the value of `api_key` from the output.


Location of your watsonx.ai Runtime instance can be retrieved in the following way:
```
ibmcloud login --apikey API_KEY -a https://cloud.ibm.com
ibmcloud resource service-instance INSTANCE_NAME
```

In result, get the value of `location` from the output.

**Tip**: Your `Cloud API key` can be generated by going to the [**Users** section of the Cloud console](https://cloud.ibm.com/iam#/users). From that page, click your name, scroll down to the **API Keys** section, and click **Create an IBM Cloud API key**. Give your key a name and click **Create**, then copy the created key and paste it below. You can also get a service specific url by going to the [**Endpoint URLs** section of the watsonx.ai Runtime docs](https://cloud.ibm.com/apidocs/machine-learning).  You can check your instance location in your  <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance details.

You can also get service specific apikey by going to the [**Service IDs** section of the Cloud Console](https://cloud.ibm.com/iam/serviceids).  From that page, click **Create**, then copy the created key and paste it below.

**Action**: Enter your `url` and `api_key` in the following cell.

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
)

In [3]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Working with spaces

You need to create a space that will be used for your work. If you do not have a space, you can use [Deployment Spaces Dashboard](https://dataplatform.cloud.ibm.com/ml-runtime/spaces?context=cpdaas) to create one.

- Click **New Deployment Space**
- Create an empty space
- Select Cloud Object Storage
- Select watsonx.ai Runtime instance and press **Create**
- Copy `space_id` and paste it below

**Tip**: You can also use SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: assign space ID below


In [4]:
space_id = "PASTE YOUR SPACE ID HERE"

You can use the `list` method to print all existing spaces.

In [ ]:
client.spaces.list(limit=10)

To be able to interact with all resources available in watsonx.ai Runtime, you need to set the **space** which you will be using.

In [5]:
client.set.default_space(space_id)

'SUCCESS'

<a id="2.-Load-historical-experiment"></a>
## 2. Load historical experiment

Initialiaze AutoAI experiment with watsonx.ai Runtime credentials and space.  

In [6]:
from ibm_watsonx_ai.experiment import AutoAI

experiment = AutoAI(credentials, space_id=space_id)

List all previous AutoAI experiment runs named `'Credit Risk Prediction and bias detection - AutoAI'` which was run in sample notebook <a href="https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/experiments/autoai/fairness/Use%20AutoAI%20to%20train%20fair%20models.ipynb" target="_blank" rel="noopener no referrer">"Use AutoAI to train fair models"</a>. 

**NOTE:** If you don't have any experiment listed below please run the <a href="https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/experiments/autoai/fairness/Use%20AutoAI%20to%20train%20fair%20models.ipynb" target="_blank" rel="noopener no referrer">"Use AutoAI to train fair models"</a> notebook first and then continue running the current notebook. 

In [7]:
autoai_experiment_name = "Credit Risk Prediction and bias detection - AutoAI"

historical_experiments = experiment.runs(filter=autoai_experiment_name).list()
historical_experiments.head()

,timestamp,run_id,state,auto_pipeline_optimizer name
0,2026-02-12T13:01:35.622Z,cfa31ea8-5bbd-45b5-933f-bc8ed107a2df,completed,Credit Risk Prediction and bias detection - Au...


Load last experiment run to variable `pipeline_optimizer`.

In [8]:
run_id = historical_experiments.run_id[0]

pipeline_optimizer = experiment.runs.get_optimizer(run_id)

In [9]:
summary = pipeline_optimizer.summary()
summary

,Enhancements,Estimator,training_accuracy_and_disparate_impact_(optimized),training_disparate_impact_Sex,training_roc_auc,holdout_disparate_impact_Sex,holdout_average_precision,holdout_log_loss,holdout_roc_auc,training_disparate_impact,...,holdout_accuracy,holdout_balanced_accuracy,training_recall,holdout_f1,training_accuracy,holdout_disparate_impact,training_balanced_accuracy,holdout_disparate_impact_Age,training_f1,training_disparate_impact_Age
Pipeline Name,,,,,,,,,,,,,,,,,,,,,
Pipeline_1,,XGBClassifier,0.676887,1.009581,0.846120,1.046499,0.480936,0.419151,0.855620,1.825512,...,0.811623,0.754275,0.894970,0.867606,0.796567,1.431694,0.748223,1.426056,0.853965,2.329145
Pipeline_2,HPO,XGBClassifier,0.676887,1.009581,0.846120,1.046499,0.480936,0.419151,0.855620,1.825512,...,0.811623,0.754275,0.894970,0.867606,0.796567,1.431694,0.748223,1.426056,0.853965,2.329145
Pipeline_3,"HPO, FE",XGBClassifier,0.681126,1.009843,0.846576,1.057825,0.481095,0.416944,0.855187,1.787199,...,0.809619,0.755745,0.892283,0.865248,0.795004,1.463687,0.747200,1.451613,0.852646,2.286728
Pipeline_4,"HPO, FE, HPO",XGBClassifier,0.681126,1.009843,0.846576,1.057825,0.481095,0.416944,0.855187,1.787199,...,0.809619,0.755745,0.892283,0.865248,0.795004,1.463687,0.747200,1.451613,0.852646,2.286728
Pipeline_5,"HPO, FE, HPO, Ensemble",BatchedTreeEnsembleClassifier(XGBClassifier),0.681126,1.009843,0.846576,1.057825,0.481095,0.416944,0.855187,1.787199,...,0.809619,0.755745,0.892283,0.865248,0.795004,1.463687,0.747200,1.451613,0.852646,2.286728


### Get selected pipeline model

Download pipeline model object from the AutoAI training job.

In [10]:
best_pipeline = pipeline_optimizer.get_pipeline()

  Using cached pyarrow-23.0.0-cp312-cp312-macosx_12_0_x86_64.whl.metadata (3.0 kB)
Using cached pyarrow-23.0.0-cp312-cp312-macosx_12_0_x86_64.whl (35.8 MB)


### Get Credit Risk dataset from experiment configuration. 

In [11]:
data_connections = pipeline_optimizer.get_data_connections()

X_train, X_holdout, y_train, y_holdout = data_connections[0].read(
    with_holdout_split=True
)

X_holdout.head()

,CheckingStatus,LoanDuration,CreditHistory,LoanPurpose,LoanAmount,ExistingSavings,EmploymentDuration,InstallmentPercent,Sex,OthersOnLoan,CurrentResidenceDuration,OwnsProperty,Age,InstallmentPlans,Housing,ExistingCreditsCount,Job,Dependents,Telephone,ForeignWorker
13,0_to_200,29,credits_paid_to_date,furniture,3705,less_100,less_1,3,female,co-applicant,3,car_other,44,none,own,1,skilled,1,none,yes
31,no_checking,26,prior_payments_delayed,appliances,4391,500_to_1000,greater_7,4,female,none,3,car_other,39,stores,own,2,unskilled,2,yes,yes
3673,less_0,20,credits_paid_to_date,retraining,4683,less_100,4_to_7,2,male,none,1,savings_insurance,46,stores,free,1,skilled,1,yes,yes
3611,no_checking,25,prior_payments_delayed,car_used,5426,500_to_1000,4_to_7,4,male,none,3,car_other,41,none,own,1,skilled,1,none,yes
433,greater_200,34,prior_payments_delayed,furniture,5470,100_to_500,4_to_7,4,male,none,4,car_other,33,none,own,2,skilled,1,none,no


<a id="3.-Bias-detection-and-mitigation"></a>
## 3. Bias detection and mitigation

The `fairness_info` dictionary contains some fairness-related metadata. The favorable and unfavorable label are values of the target class column that indicate whether the loan was granted or denied. A protected attribute is a feature that partitions the population into groups whose outcome should have parity. The credit-risk dataset has two protected attribute columns, sex and age. Each prottected attributes has monitored and reference group.


In [12]:
fairness_info = pipeline_optimizer.get_params()["fairness_info"]
fairness_info

{'favorable_labels': ['No Risk'],
 'protected_attributes': [{'feature': 'Sex',
   'monitored_group': ['female'],
   'reference_group': ['male']},
  {'feature': 'Age',
   'monitored_group': [[18, 25]],
   'reference_group': [[26, 75]]}],
 'unfavorable_labels': ['Risk']}

### Calculate fairness metrics

We will calculate some model metrics. Accuracy describes how accurate is the model according to dataset. 
Disparate impact is defined by comparing outcomes between a privileged group and an unprivileged group, 
so it needs to check the protected attribute to determine group membership for the sample record at hand. The closer to 1 is the value of disparate impact the less biased is the model. 
The third calculated metric takes the disparate impact into account along with accuracy. The best value of the score is 1.0.

In [13]:
import sklearn.metrics
from lale.lib.aif360 import accuracy_and_disparate_impact, disparate_impact

accuracy_scorer = sklearn.metrics.make_scorer(sklearn.metrics.accuracy_score)
print(f"accuracy {accuracy_scorer(best_pipeline, X_holdout, y_holdout):.1%}")

disparate_impact_scorer = disparate_impact(**fairness_info)
print(
    f"disparate impact {disparate_impact_scorer(best_pipeline, X_holdout, y_holdout):.2f}"
)

combined_scorer = accuracy_and_disparate_impact(**fairness_info)
print(
    f"accuracy and disparate impact metric {combined_scorer(best_pipeline, X_holdout, y_holdout):.2f}"
)

accuracy 81.2%
disparate impact 1.43
accuracy and disparate impact metric 0.76


<a id="4.-Refinery-with-lale"></a>
## 4. Refinery with lale

In this section we will use `DisparateImpactRemover` algorithm for mitigating fairness problems from `lale.lib.aif360` module. It modifies the features that are not the protected attribute in such a way that it is hard to predict the protected attribute from them. This algorithm has a hyperparameter `repair_level` that we will tune with hyperparameter optimization.

In [14]:
from lale.lib.aif360 import DisparateImpactRemover
from lale.pretty_print import ipython_display

ipython_display(DisparateImpactRemover.hyperparam_schema("repair_level"))

```python
{
    "description": "Repair amount from 0 = none to 1 = full.",
    "type": "number",
    "minimum": 0,
    "maximum": 1,
    "default": 1,
}
```

### Pipeline decomposition and new definition

Start by removing the last step of the pipeline, i.e., the final estimator.

In [15]:
prefix = best_pipeline.remove_last().freeze_trainable()
prefix.export_to_sklearn_pipeline()

Pipeline(steps=[('featureunion',
                 FeatureUnion(transformer_list=[('float32_transform_6378807152',
                                                 Pipeline(steps=[('numpycolumnselector',
                                                                  NumpyColumnSelector(columns=[0,
                                                                                               1,
                                                                                               2,
                                                                                               3,
                                                                                               5,
                                                                                               6,
                                                                                               7,
                                                                                               8,
                                                                                               9,
                                                                                               10,
                                                                                               11,
                                                                                               12,
                                                                                               13,
                                                                                               14,
                                                                                               15,
                                                                                               16,
                                                                                               17,
                                                                                               18,
                                                                                               19])),
                                                                 ('compressstrings',
                                                                  CompressStrings(compress_type='hash',
                                                                                  dtypes_list=['char_str',
                                                                                               'int_num',
                                                                                               'char_str',
                                                                                               'char_str',
                                                                                               'char_str',
                                                                                               'char_str',
                                                                                               'i...
                                                                 ('numpyreplacemissingvalues',
                                                                  NumpyReplaceMissingValues(missing_values=[])),
                                                                 ('numimputer',
                                                                  NumImputer(missing_values=nan,
                                                                             strategy='median')),
                                                                 ('optstandardscaler',
                                                                  OptStandardScaler(use_scaler_flag=False)),
                                                                 ('float32_transform',
                                                                  float32_transform())]))])),
                ('numpypermutearray',
                 NumpyPermuteArray(axis=0,
                                   permutation_ind

Initialize the `DisparateImpactRemover` with fairness configuration and pipeline without final estimator and add a new final step, which consists of a choice of two estimators. In this code, `|` is the or combinator (algorithmic choice). It defines a search space for another optimizer run.

In [16]:
from lale.operator_wrapper import wrap_imported_operators
from sklearn.ensemble import RandomForestClassifier as RF
from sklearn.linear_model import LogisticRegression as LR

wrap_imported_operators()

di_remover = DisparateImpactRemover(**fairness_info, preparation=prefix)
planned_fairer = di_remover >> (LR | RF)

Fairness metrics can be more unstable than accuracy, because they depend not just on the distribution of labels, but also on the distribution of privileged and unprivileged groups as defined by the protected attributes. In AI Automation, k-fold cross validation helps reduce overfitting. To get more stable results, we will stratify these k folds by both labels and groups with `FairStratifiedKFold` class.

In [17]:
from lale.lib.aif360 import FairStratifiedKFold

fair_cv = FairStratifiedKFold(**fairness_info, n_splits=3)

### Pipeline training

To automatically select the algorithm and tune its hyperparameters we use `auto_configure` method of lale pipeline. The combined metric `accuracy_and_disparate_impact` is used as scoring metric in evaluation process.   

In [18]:
from lale.lib.lale import Hyperopt

trained_fairer = planned_fairer.auto_configure(
    X_train,
    y_train,
    optimizer=Hyperopt,
    cv=fair_cv,
    max_evals=10,
    scoring=combined_scorer,
    best_score=1.0,
)

100%|██████████| 10/10 [00:18<00:00,  1.81s/trial, best loss: 0.1676332233430191]


#### Results 

Visualize the final pipeline and calculate its metrics.  

In [19]:
trained_fairer.export_to_sklearn_pipeline()

Pipeline(steps=[('_disparateimpactremoverimpl',
                 <lale.lib.aif360.disparate_impact_remover._DisparateImpactRemoverImpl object at 0x17bc8e6c0>),
                ('randomforestclassifier',
                 RandomForestClassifier(criterion='entropy',
                                        min_samples_leaf=0.3551854079747729,
                                        min_samples_split=0.3003021725296457,
                                        n_estimators=29))])

In [20]:
print(f"accuracy {accuracy_scorer(trained_fairer, X_holdout, y_holdout):.1%}")
print(
    f"disparate impact {disparate_impact_scorer(trained_fairer, X_holdout, y_holdout):.2f}"
)
print(
    f"accuracy and disparate impact metric {combined_scorer(trained_fairer, X_holdout, y_holdout):.2f}"
)

accuracy 66.5%
disparate impact 1.00
accuracy and disparate impact metric 0.83


**Summary:** As result of the described steps we received unbiased pipeline model based on disparate impact value, however the accuracy of the model decreased from 70% to 66%.

<a id="5.-Deploy-and-score"></a>
## 5. Deploy and score
In this section you will learn how to deploy and score Lale pipeline model using watsonx.ai Runtime instance.

### Store the model

In [21]:
model_props = {client.repository.ModelMetaNames.NAME: "Fairer AutoAI model"}
feature_vector = list(X_train.columns)

In [22]:
published_model = client.repository.store_model(
    model=best_pipeline.export_to_sklearn_pipeline(),
    meta_props=model_props,
    training_id=run_id,
)

In [23]:
published_model_id = client.repository.get_model_id(published_model)

### Deployment creation

In [24]:
metadata = {
    client.deployments.ConfigurationMetaNames.NAME: "Deployment of fairer model",
    client.deployments.ConfigurationMetaNames.ONLINE: {},
}

created_deployment = client.deployments.create(published_model_id, meta_props=metadata)



######################################################################################

Synchronous deployment creation for id: 'f5770ff4-bf8d-4ee9-b539-266d6a1030e9' started

######################################################################################


initializing
Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.
.....
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='523d3945-9f09-4b0e-addf-08e1253f0568'
-----------------------------------------------------------------------------------------------




In [25]:
deployment_id = client.deployments.get_id(created_deployment)

#### Deployment scoring 

You need to pass scoring values as input data if the deployed model. Use `client.deployments.score()` method to get predictions from deployed model. 

In [26]:
values = X_holdout.values

scoring_payload = {"input_data": [{"values": values[:5]}]}

In [27]:
predictions = client.deployments.score(deployment_id, scoring_payload)
predictions

{'predictions': [{'fields': ['prediction', 'probability'],
   'values': [['No Risk', [0.6803585290908813, 0.31964150071144104]],
    ['Risk', [0.2891402840614319, 0.7108597159385681]],
    ['No Risk', [0.943277895450592, 0.05672210082411766]],
    ['No Risk', [0.7675577402114868, 0.23244228959083557]],
    ['No Risk', [0.7859511971473694, 0.21404878795146942]]]}]}

<a id="6.-Cleanup"></a>
## 6. Cleanup

If you want to clean up all created assets:
- experiments
- trainings
- pipelines
- model definitions
- models
- functions
- deployments

please follow up this sample [notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="7.-Summary-and-next-steps"></a>
## 7. Summary and next steps

 You successfully completed this notebook!

Check out used packeges domuntations:
- `ibm-watsonx-ai` [Online Documentation](https://www.ibm.com/cloud/watson-studio/autoai)
- `lale`: https://github.com/IBM/lale
- `aif360`: https://aif360.mybluemix.net/

### Authors 

**Dorota Lączak**, Software Engineer at watsonx.ai

**Mateusz Szewczyk**, Software Engineer at watsonx.ai

Copyright © 2022-2026 IBM. This notebook and its source code are released under the terms of the MIT License.